In [1]:
!pip install -q -U faiss-cpu sentence-transformers transformers peft accelerate boto3

import os, json, boto3, faiss
from functools import lru_cache
from google.colab import userdata
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from sentence_transformers import CrossEncoder

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 115.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 105.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 10.8 MB/s eta 0:00:00


In [2]:
!pip install -U torchao
import torch
import torchao
import peft
import re

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 42.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [3]:
BUCKET = "legai-ai-embeddings"
FAISS_KEY = "faiss/faiss_rebuilt.index"
ADAPTER_PREFIX = "finetune/final_adapter/"
LOCAL_FAISS = "/content/faiss_rebuilt.index"
LOCAL_ADAPTER = "/content/legai_final_adapter"
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
EMBED_MODEL = "BAAI/bge-base-en-v1.5"
s3 = boto3.client("s3", aws_access_key_id=userdata.get("AWS_ACCESS_KEY_ID").strip(), aws_secret_access_key=userdata.get("AWS_SECRET_ACCESS_KEY").strip(), region_name="us-east-1")

In [4]:
s3.download_file(BUCKET, FAISS_KEY, LOCAL_FAISS)
index = faiss.read_index(LOCAL_FAISS)
print("Vectors:", index.ntotal)
print("Dimension:", index.d)

Vectors: 4956398
Dimension: 384


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
embed_model = SentenceTransformer(EMBED_MODEL, device=device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
def embed_query(question):
    embedding = embed_model.encode([question], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    return embedding[:, :384]

In [7]:
metadata_keys = []

for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix="metadata/"):

    metadata_keys.extend([obj["Key"] for obj in page.get("Contents", []) if obj["Key"].endswith(".jsonl")])

metadata_keys.sort(key=lambda x: int(x.split("metadata_")[-1].split(".jsonl")[0]))

print("Metadata files:", len(metadata_keys))

Metadata files: 7015


In [26]:
@lru_cache(maxsize=7000)
def load_metadata_file(key):
    obj = s3.get_object(Bucket=BUCKET, Key=key)
    return [json.loads(line) for line in obj["Body"].read().decode("utf-8").splitlines() if line.strip()]

In [24]:
def get_metadata(vector_id):
    target = int(vector_id)
    left, right = 0, len(metadata_keys) - 1

    while left <= right:
        mid = (left + right) // 2
        rows = load_metadata_file(metadata_keys[mid])

        if not rows:
            return None

        first_id, last_id = int(rows[0]["vector_id"]), int(rows[-1]["vector_id"])

        if target < first_id:
            right = mid - 1
        elif target > last_id:
            left = mid + 1
        else:
            for row in rows:
                if int(row["vector_id"]) == target:
                    return row
            return None

    return None

In [8]:
reranker = CrossEncoder("BAAI/bge-reranker-base", max_length=512, device=device)

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

In [9]:
def retrieve_candidates(question, k=20):
    query_vector = embed_query(question)
    distances, ids = index.search(query_vector, k)
    results = []

    for distance, vector_id in zip(distances[0], ids[0]):

        if vector_id == -1:
            continue

        metadata = get_metadata(vector_id)

        if metadata:
            results.append({"vector_id": int(vector_id), "distance": float(distance), "source": metadata.get("source"), "source_file": metadata.get("source_file"), "text": metadata["text"]})

    return results

In [10]:
def rerank(question, results):
    if not results:
        return []

    pairs = [[question, r["text"]] for r in results]
    scores = reranker.predict(pairs)

    for r, score in zip(results, scores):
        r["rerank_score"] = float(score)

    return sorted(results, key=lambda x: x["rerank_score"], reverse=True)

In [11]:
def filter_evidence(results, min_score=0.90, max_docs=5):
    strong = [r for r in results if r["rerank_score"] >= min_score]
    return strong[:max_docs]

In [ ]:
def split_sentences(text): return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

In [ ]:
def get_best_evidence(question, results, top_n=3, min_score=0.90):

    sentences = []

    for i, r in enumerate(results, 1):
        for sentence in split_sentences(r["text"]):

            if len(sentence) > 30:
                sentences.append({"document": i, "text": sentence})

    if not sentences:
        return []

    scores = reranker.predict([[question, s["text"]] for s in sentences])

    for s, score in zip(sentences, scores):
        s["score"] = float(score)

    sentences = sorted(sentences, key=lambda x: x["score"], reverse=True)
    return [s for s in sentences if s["score"] >= min_score][:top_n]

In [12]:
# def retrieve(question, candidate_k=20, min_score=0.90, max_docs=5):
#     candidates = retrieve_candidates(question, candidate_k)
#     ranked = rerank(question, candidates)
#     return filter_evidence(ranked, min_score, max_docs)

# def retrieve(question, candidate_k=20, max_docs=5):
#     candidates = retrieve_candidates(question, candidate_k)
#     ranked = rerank(question, candidates)
#     return ranked[:max_docs]

def retrieve(question, candidate_k=20, min_score=0.70, max_docs=5):
    candidates = retrieve_candidates(question, candidate_k)
    ranked = rerank(question, candidates)
    return filter_evidence(ranked, min_score=min_score, max_docs=max_docs)

In [ ]:
question = "What notice applies when a lease is terminated for nonpayment of rent?"

results = retrieve(question)
evidence = get_best_evidence(question, results, top_n=3)

for i, e in enumerate(evidence, 1):
    print(f"\n--- EVIDENCE {i} | Score: {e['score']:.4f} | Document: {e['document']} ---")
    print(e["text"])

In [ ]:
for i, r in enumerate(results, 1):
    print(f"\n--- RESULT {i} | Distance: {r['distance']:.4f} ---")
    print(r["text"][:1500])

In [13]:
os.makedirs(LOCAL_ADAPTER, exist_ok=True)

for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=ADAPTER_PREFIX):
    for obj in page.get("Contents", []):
        key = obj["Key"]

        if key.endswith("/"):
            continue

        relative = key[len(ADAPTER_PREFIX):]
        local_path = os.path.join(LOCAL_ADAPTER, relative)
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        s3.download_file(BUCKET, key, local_path)
print("Adapter downloaded.")

Adapter downloaded.


In [14]:
tokenizer = AutoTokenizer.from_pretrained(LOCAL_ADAPTER)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto")

model = PeftModel.from_pretrained(base_model, LOCAL_ADAPTER)
model.eval()
model.config.use_cache = True

print("Fine-tuned LegaiAI loaded.")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Fine-tuned LegaiAI loaded.


In [15]:
def build_context(results):
    return "\n\n".join([f"[Document {i}]\n{r['text']}" for i, r in enumerate(results, 1)])

In [35]:
SYSTEM = """
You are LegaiAI, a legal information assistant.

Answer the user's question using only the provided legal context.

Your goal is to provide the most accurate, complete, and legally faithful answer supported by the retrieved context.

Carefully examine all retrieved passages before deciding whether the answer is available.

If at least one retrieved passage directly contains the rule, condition, deadline, amount, actor, legal consequence, exception, or procedure needed to answer the question, do not state that the context is insufficient.

Use the directly relevant passage or passages and ignore unrelated retrieved passages.

Preserve the exact legal meaning of the source material.

Do not change:
- who performs an action
- what action occurs
- when the action occurs
- numbers
- monetary amounts
- dates
- times
- deadlines
- durations
- notice periods
- section numbers
- legal conditions
- exceptions
- triggers
- limitations
- consequences

Pay particular attention to distinctions such as:
- before vs. after
- same day vs. next day
- may vs. must
- shall vs. may
- and vs. or
- more than vs. at least
- permitted vs. prohibited
- included vs. excluded
- rolled over vs. not rolled over
- certified vs. not certified
- calculated by one party vs. transmitted by another party

If the context states an exact number, amount, time, deadline, multiplier, legal condition, or legal consequence, preserve it exactly.

If a calculation is required and the governing formula or rule is explicitly supported by the context:
1. identify the controlling formula or rule,
2. perform the calculation carefully,
3. preserve all required multipliers, limits, and conditions,
4. give the final result clearly.

When multiple passages are relevant, synthesize them only when they are consistent with each other.

Do not use unrelated passages merely because they contain similar words or legal terminology.

Do not introduce legal requirements, citations, exceptions, procedures, interpretations, or factual details that are not supported by the provided context.

Do not rely on general legal knowledge, memory, or assumptions outside the supplied context.

If the retrieved context contains the answer only partially, answer only the supported portion and clearly state what specific part cannot be determined from the supplied context.

Only state that the retrieved context does not provide enough information when none of the retrieved passages contain enough legally relevant information to answer the material part of the question.

Give a direct and complete answer.

Do not add unnecessary background information.

Do not quote large portions of the context unless necessary to preserve an exact legal rule.

Before producing the final answer, internally verify:
- the correct actor
- the correct action
- the correct timing
- the correct number or amount
- the correct condition
- the correct exception
- the correct legal consequence

Then provide only the final answer.
"""

In [ ]:
# def ask_legai(context, question):

#     messages = [{"role": "system", "content": SYSTEM},
#      {"role": "user", "content": f"LEGAL CONTEXT:\n{context}\n\nQUESTION:\n{question}\n\nAnswer concisely using only explicitly stated information."}]

#     prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

#     inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to(model.device)

#     with torch.inference_mode():
#         outputs = model.generate(**inputs, max_new_tokens=180, do_sample=False, repetition_penalty=1.1, no_repeat_ngram_size=4, pad_token_id=tokenizer.eos_token_id)

#     generated = outputs[0][inputs["input_ids"].shape[1]:]
#     return tokenizer.decode(generated, skip_special_tokens=True).strip()

In [36]:
def ask_legai(context, question):

    messages = [
        {
            "role": "system",
            "content": SYSTEM
        },
        {
            "role": "user",
            "content": f"""
LEGAL CONTEXT:

{context}

QUESTION:

{question}

Answer the question using only the legal context above.

Use directly relevant passages and ignore unrelated passages.

If the context contains the necessary rule, do not abstain.

Preserve exact legal meaning, especially actors, amounts, deadlines, conditions, exceptions, calculations, and consequences.

FINAL ANSWER:
"""
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(model.device)

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    return answer

In [ ]:
def is_multi_rule_question(question):
    q = question.lower()
    keywords = ["compare", "difference", "different", "both", " and ", "periods", "rules", "requirements"]
    return any(k in q for k in keywords)

def clean_evidence_text(text):
    text = text.replace("Source:", "").strip()
    text = text.replace("Effective Date Note:", "").strip()
    text = text.replace("Civilian", "").strip()
    return text

In [37]:
def rag(
    question,
    candidate_k=20,
    min_score=0.70,
    max_docs=5
):

    results = retrieve(
        question,
        candidate_k=candidate_k,
        min_score=min_score,
        max_docs=max_docs
    )

    if not results:

        return (
            "The retrieved legal context does not provide enough information to answer this question.",
            []
        )

    context = build_context(results)

    answer = ask_legai(
        context=context,
        question=question
    )

    return answer, results

#downloaded fixed JSONL back into Colab

In [54]:
FIXED_EVAL_FILE = "/content/legai_fixed_eval_50_v2.jsonl"

fixed_records = []

with open(
    FIXED_EVAL_FILE,
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        if line.strip():
            fixed_records.append(
                json.loads(line)
            )

print(
    "Loaded fixed evaluation cases:",
    len(fixed_records)
)

Loaded fixed evaluation cases: 50


In [ ]:
# print((evidence))

In [ ]:
# question = "What notice periods apply to voluntary lease termination and termination for nonpayment of rent?"

# answer, evidence = rag(question)

# print("ANSWER:")
# print(answer)

# print("\nEVIDENCE:")

# for i, e in enumerate(evidence, 1):
#     print(f"\n--- EVIDENCE {i} | Rerank Score: {e['rerank_score']:.4f} ---")
#     print(e["text"])

In [ ]:
# question = "How many days of written notice may a family give to terminate the lease?"
# answer, evidence = rag(question)

# print("ANSWER:")
# print(answer)

# print("\nEVIDENCE:")

# for i, e in enumerate(evidence, 1):
#     print(f"\n--- EVIDENCE {i} | Score: {e['rerank_score']:.4f}---")
#     print(e["text"])

# ***EVALUATION***

Question
   
1. Retrieval evaluation: This checks whether FAISS retrieved the correct legal document/chunk. Recall, Precision, MRR
   
2. Reranker / evidence evaluation: Evidence Hit@1, Evidence Hit@3, MRR
   
3. Answer evaluation: Answer correctness, Faithfulness, Numerical accuracy, Completeness
   
4. Abstention / safety evaluation:
True Positive:
Answer exists → system answers

True Negative:
Answer missing → system refuses correctly

False Positive:
Answer missing → system invents answer ❌

False Negative:
Answer exists → system refuses ❌

In [20]:
!pip install -q -U deepeval openai

import os
import json
import random
import hashlib

from google.colab import userdata
from botocore.exceptions import ClientError

from deepeval.models import AzureOpenAIModel
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig, DisplayConfig, ErrorConfig

from deepeval.metrics import (
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    ContextualRelevancyMetric,
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    GEval
)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.29.0 requires click<9.0.0,>=8.4.2, but you have click 8.3.3 which is incompatible.


In [21]:
azure_endpoint = userdata.get("AZURE_OPENAI_ENDPOINT").strip().rstrip("/") + "/"
azure_deployment = userdata.get("AZURE_OPENAI_DEPLOYMENT").strip()
judge_model_name = userdata.get("AZURE_OPENAI_MODEL").strip()
azure_api_key = userdata.get("AZURE_OPENAI_API_KEY").strip()

judge = AzureOpenAIModel(
    model=judge_model_name,
    deployment_name=azure_deployment,
    api_key=azure_api_key,
    api_version="2025-04-01-preview",
    base_url=azure_endpoint,
    temperature=0
)

print(judge.generate("Reply only with READY"))

('READY', 0.00011250000000000001)


In [55]:
os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "240"
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "900"
os.environ["DEEPEVAL_RETRY_MAX_ATTEMPTS"] = "2"

# 3. EXCLUDE FINE-TUNING CONTEXTS FROM EVALUATION

In [ ]:

TRAIN_SAMPLE_KEY = "finetune/sampled_contexts_5000.jsonl"

def text_hash(text):
   return hashlib.sha256( text.strip().encode("utf-8") ).hexdigest()


def load_training_hashes():

    hashes = set()

    try:

        obj = s3.get_object( Bucket=BUCKET, Key=TRAIN_SAMPLE_KEY )

        lines = obj["Body"].read().decode("utf-8").splitlines()

        for line in lines:

            if not line.strip():
                continue

            row = json.loads(line)

            text = row.get("text") or row.get("context")

            if text:
                hashes.add(text_hash(text))

    except ClientError as e:

        code = e.response["Error"]["Code"]

        if code in ["NoSuchKey", "404"]:
            print("Training sample file not found. No training exclusion applied.")
        else:
            raise

    return hashes


training_hashes = load_training_hashes()

print("Fine-tuning contexts excluded:", len(training_hashes))


Fine-tuning contexts excluded: 5000


# 4. SAMPLE 50 HELD-OUT LEGAL CONTEXTS

In [ ]:
def sample_heldout_contexts( n=50, min_chars=200, max_chars=3500, seed=42):

    rng = random.Random(seed)

    keys = metadata_keys.copy()
    rng.shuffle(keys)

    contexts = []
    selected_info = []
    seen = set()

    for key in keys:

        rows = list(load_metadata_file(key))
        rng.shuffle(rows)

        for row in rows:

            text = row.get("text", "").strip()

            if len(text) < min_chars or len(text) > max_chars:
                continue

            h = text_hash(text)

            if h in training_hashes:
                continue

            if h in seen:
                continue

            seen.add(h)

            contexts.append([text])

            selected_info.append({
                "vector_id": int(row["vector_id"]),
                "source": row.get("source"),
                "source_file": row.get("source_file"),
                "metadata_key": key
            })

            if len(contexts) >= n:
                return contexts, selected_info

    return contexts, selected_info


N_CONTEXTS = 50

eval_contexts, eval_context_info = sample_heldout_contexts(
    n=N_CONTEXTS
)

print("Held-out evaluation contexts:", len(eval_contexts))

Held-out evaluation contexts: 50


# 5. GENERATE GOLDEN QUESTIONS + EXPECTED ANSWERS

In [ ]:
from deepeval.synthesizer import Synthesizer
from deepeval.synthesizer.config import StylingConfig

styling = StylingConfig(
    task="Evaluate a legal retrieval-augmented question answering system.",
    scenario="A user asks factual legal questions that must be answered from supplied legal material.",
    input_format="A clear legal question whose answer is explicitly supported by the supplied context.",
    expected_output_format=(
        "A concise and complete factual answer grounded only in the supplied legal context. "
        "Preserve exact numbers, dates, durations, notice periods, section numbers, "
        "parties, actions, conditions, exceptions, and legal relationships."
    )
)

synthesizer = Synthesizer( model=judge, styling_config=styling, async_mode=False )

goldens = synthesizer.generate_goldens_from_contexts( contexts=eval_contexts, include_expected_output=True, max_goldens_per_context=1 )

print("Generated goldens:", len(goldens))


Output()

Generated goldens: 50


# 6. INSPECT FIRST 5 GOLDENS

In [ ]:
for i, golden in enumerate(goldens[:1], 1):

    print(f"\n========== GOLDEN {i} ==========")

    print("\nQUESTION:")
    print(golden.input)

    print("\nEXPECTED ANSWER:")
    print(golden.expected_output)

    print("\nSOURCE CONTEXT:")
    print(
        golden.context[0][:1200]
        if golden.context
        else "No context"
    )


# 7. SAVE FIXED EVALUATION SET

In [ ]:
FIXED_EVAL_FILE = "/content/legai_fixed_eval_50_v2.jsonl"

with open(FIXED_EVAL_FILE, "w", encoding="utf-8") as f:

    for i, golden in enumerate(goldens):

        info = eval_context_info[i]

        record = {
            "test_case": i,
            "input": golden.input,
            "expected_output": golden.expected_output,
            "context": golden.context,
            "vector_id": info["vector_id"],
            "source": info.get("source"),
            "source_file": info.get("source_file"),
            "metadata_key": info.get("metadata_key")
        }

        f.write( json.dumps( record, ensure_ascii=False ) + "\n" )

print("Saved fixed evaluation set:", FIXED_EVAL_FILE)

# 8. RUN YOUR NEW GENERATIVE RAG ON ALL 50 QUESTIONS for new goldens


In [ ]:
# test_cases = []
# rag_run_records = []

# for i, golden in enumerate(goldens, 1):

#     question = golden.input

#     print( f"Running RAG {i}/{len(goldens)}: " f"{question[:100]}" )

#     try:

#         answer, evidence = rag(question)

#         retrieval_context = [ e["text"] for e in evidence ]

#         if not retrieval_context:
#             retrieval_context = [ "NO_RELEVANT_CONTEXT_RETRIEVED" ]

#         test_case = LLMTestCase(
#             input=question,
#             actual_output=answer,
#             expected_output=golden.expected_output,
#             retrieval_context=retrieval_context,
#             context=golden.context
#         )

#         test_cases.append(test_case)

#         rag_run_records.append({
#             "question": question,
#             "expected_answer": golden.expected_output,
#             "actual_answer": answer,
#             "retrieval_context": retrieval_context,
#             "source_context": golden.context
#         })

#     except Exception as e:

#         print(
#             f"ERROR on test {i}:",
#             e
#         )


# print("Answer test cases ready:", len(test_cases))

Saved goldens

In [56]:
test_cases = []
rag_run_records = []

for i, row in enumerate(fixed_records, 1):

    question = row["input"]
    expected_answer = row["expected_output"]
    source_context = row["context"]

    print(
        f"Running RAG {i}/{len(fixed_records)}: "
        f"{question[:100]}"
    )

    try:

        answer, evidence = rag(question)

        retrieval_context = [
            e["text"]
            for e in evidence
        ]

        if not retrieval_context:
            retrieval_context = [
                "NO_RELEVANT_CONTEXT_RETRIEVED"
            ]

        test_case = LLMTestCase(
            input=question,
            actual_output=answer,
            expected_output=expected_answer,
            retrieval_context=retrieval_context,
            context=source_context
        )

        test_cases.append(
            test_case
        )

        rag_run_records.append({
            "question": question,
            "expected_answer": expected_answer,
            "actual_answer": answer,
            "retrieval_context": retrieval_context,
            "source_context": source_context
        })

    except Exception as e:

        print(
            f"ERROR on test {i}:",
            e
        )

print(
    "Answer test cases ready:",
    len(test_cases)
)

Running RAG 1/50: Under the applicable legal rules governing the Thrift Savings Plan (TSP), how do posting timelines d
Running RAG 2/50: Under federal employment and benefits law, how should I calculate the duration for which Thrift Savi
Running RAG 3/50: Under the relevant legal framework, is the Recognized Statistical Agency or Unit required to have ex
Running RAG 4/50: How does the law define Subpart B positions in situations where an entity controls or funds more tha
Running RAG 5/50: Under 5 U.S.C. § 552 and implementing regulation § 1631.8, what documentation procedures must agenci
Running RAG 6/50: Under what specific OPM and OMB appraisal-system certification conditions may Senior Executive Servi
Running RAG 7/50: Under the applicable legal framework, how do the standards governing remedial actions differ from th
Running RAG 8/50: How are non-Roth distributions from an eligible employer plan treated for federal income tax purpose
Running RAG 9/50: Under Subpart B—Employing Agen

# 9. SAVE NEW RAG OUTPUTS

In [57]:
RAG_OUTPUT_FILE = "/content/legai_rag_outputss_v2.jsonl"

with open( RAG_OUTPUT_FILE, "w", encoding="utf-8" ) as f:

    for row in rag_run_records:

        f.write( json.dumps( row, ensure_ascii=False ) + "\n" )

print("Saved RAG outputs:", RAG_OUTPUT_FILE)

Saved RAG outputs: /content/legai_rag_outputss_v2.jsonl


# 10. CREATE SEPARATE RETRIEVAL TEST CASES

In [ ]:
retrieval_test_cases = []

for i, golden in enumerate(goldens, 1):

    question = golden.input

    print( f"Running retrieval {i}/{len(goldens)}: " f"{question[:100]}" )

    try:

        candidates = retrieve_candidates( question, k=20 )

        ranked = rerank( question, candidates )

        # filtered = filter_evidence( ranked, min_score=0.90, max_docs=5 )

        # retrieval_context = [ r["text"] for r in filtered ]

        retrieval_context = [ r["text"] for r in ranked[:5] ]

        if not retrieval_context:
            retrieval_context = [ "NO_RELEVANT_CONTEXT_RETRIEVED" ]

        test_case = LLMTestCase(
            input=question,
            actual_output="Retrieval evaluation only",
            expected_output=golden.expected_output,
            retrieval_context=retrieval_context,
            context=golden.context
        )

        retrieval_test_cases.append(
            test_case
        )

    except Exception as e:

        print( f"Retrieval ERROR on test {i}:", e )

print( "Retrieval test cases ready:", len(retrieval_test_cases) )


In [ ]:
# 11. RETRIEVAL METRICS

In [29]:
# retrieval_metrics = [

#     ContextualPrecisionMetric(
#         model=judge,
#         threshold=0.70,
#         include_reason=True,
#         async_mode=False
#     ),

#     ContextualRecallMetric(
#         model=judge,
#         threshold=0.70,
#         include_reason=True,
#         async_mode=False
#     ),

#     ContextualRelevancyMetric(
#         model=judge,
#         threshold=0.70,
#         include_reason=True,
#         async_mode=False
#     )

# ]

# 12. RUN RETRIEVAL EVALUATION

In [ ]:
# retrieval_results = evaluate(
#     test_cases=retrieval_test_cases,
#     metrics=retrieval_metrics,
#     identifier="LegaiAI-Retrieval-Generative-RAG-v2",
#     hyperparameters={
#         "embedding_model": EMBED_MODEL,
#         "reranker": "BAAI/bge-reranker-base",
#         "candidate_k": 20,
#         "filtered_top_k": 5,
#         "rerank_threshold": 0.90
#     },
#     async_config=AsyncConfig(
#         run_async=False
#     ),
#     error_config=ErrorConfig(
#         ignore_errors=True
#     ),
#     display_config=DisplayConfig(
#         results_folder="/content/legai_eval_retrieval_v2"
#     )
# )


In [58]:
os.environ[
    "DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"
] = "240"

os.environ[
    "DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"
] = "900"

os.environ[
    "DEEPEVAL_RETRY_MAX_ATTEMPTS"
] = "2"

# 13. LEGAL CORRECTNESS METRIC

In [30]:
legal_correctness = GEval(
    name="Legal Correctness",
    evaluation_steps=[

        "Compare the actual output with the expected output.",

        "Check whether all material legal facts necessary to answer the question are correct.",

        "Treat a changed number, date, duration, notice period, section number, party, action, trigger, exception, or legal condition as a material error.",

        "Check whether the answer preserves who performs each action and when each action or legal consequence becomes effective.",

        "Penalize unsupported legal requirements, exceptions, interpretations, or factual additions.",

        "Penalize omission of information necessary to correctly and completely answer the user's question.",

        "Do not penalize differences in wording when the actual output preserves the same legal meaning as the expected output."

    ],
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT
    ],
    model=judge,
    threshold=0.80,
    async_mode=False
)


# 14. ANSWER METRICS

In [33]:
legal_correctness = GEval(
    name="Legal Correctness",
    evaluation_steps=[

        "Compare the actual output with the expected output.",

        "Check whether all material legal facts necessary to answer the question are correct.",

        "Treat a changed number, date, duration, notice period, section number, party, action, trigger, exception, or legal condition as a material error.",

        "Check whether the answer preserves who performs each action and when each action or legal consequence becomes effective.",

        "Penalize unsupported legal requirements, exceptions, interpretations, or factual additions.",

        "Penalize omission of information necessary to correctly and completely answer the user's question.",

        "Do not penalize differences in wording when the actual output preserves the same legal meaning as the expected output."

    ],

    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT
    ],

    model=judge,
    threshold=0.80,
    async_mode=False
)

In [59]:
answer_metrics = [

    AnswerRelevancyMetric(
        model=judge,
        threshold=0.70,
        include_reason=True,
        async_mode=False
    ),

    FaithfulnessMetric(
        model=judge,
        threshold=0.70,
        include_reason=True,
        async_mode=False
    ),

    legal_correctness
]

# 15. RUN ANSWER EVALUATION

In [60]:
answer_results = evaluate(
    test_cases=test_cases,
    metrics=answer_metrics,
    identifier="LegaiAI-Fixed50-Prompt-v2",
    hyperparameters={
        "generator_model": BASE_MODEL,
        "adapter": "finetune/final_adapter",
        "retrieval_threshold": 0.70,
        "candidate_k": 20,
        "max_docs": 5,
        "answer_mode": "generative RAG with stricter legal accuracy prompt",
        "judge_model": judge_model_name
    },
    async_config=AsyncConfig(
        run_async=False
    ),
    error_config=ErrorConfig(
        ignore_errors=True
    ),
    display_config=DisplayConfig(
        results_folder="/content/legai_eval_fixed50_prompt_v2"
    )
)

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-5.1 (Azure), strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-5.1 (Azure), strict=False, async_mode=False)...

✨ You're running DeepEval's latest Legal Correctness [GEval] Metric! (using gpt-5.1 (Azure), strict=False, 
async_mode=False)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:              Under the applicable legal rules governing the Thrift Savings Plan (TSP), how do     │
│  │                         posting timelines differ between regular plan transactions (such as                  │
│  │                         contributions, withdrawals, or loans) and investment election requests (such as      │
│  │                         changes in fund allocation), and what specific provisions in the supplied legal      │
│  │                         materials establish these differing timelines?                                       │
│  │     Actual Output:      Regular plan transactions (such as contributions, withdrawals, or loans) are         │
│  │                         posted on the same business day if entered before 12 noon Eastern Time, otherwise    │
│  │                         on the next business day. Investment election requests are posted immediately and    │
│  │                         become effective the next business day. These timelines are established by the       │
│  │                         rules stated in the first paragraph of the document.                                 │
│  │     Expected Output:    Under the supplied TSP rules, regular transaction requests (i.e., any transaction    │
│  │                         request other than an investment election request, such as contributions,            │
│  │                         withdrawals, or loans) and investment election requests have different posting       │
│  │                         timelines, as established in Source (1):                                             │
│  │                                                                                                              │
│  │                         1. **Regular transaction requests (non–investment election requests)**               │
│  │                            - If entered into the TSP record keeping system via the TSP website or the        │
│  │                         ThriftLine **before 12 noon eastern time of any business day**, they will            │
│  │                         **ordinarily be posted that business day**.                                          │
│  │                            - If entered **at or after 12 noon eastern time of any business day**, they       │
│  │                         will **ordinarily be posted on the next business day**.                              │
│  │                                                                                                              │
│  │                         2. **Investment election requests**                                                  │
│  │                            - A transaction request that **is an investment election request** will           │
│  │                         **ordinarily be posted immediately** and will **be effective the next business       │
│  │                         day**.                                                                               │
│  │                                                                                                              │
│  │                         These differing timelines are expressly set out in Source (1), which                 │
│  │                         distinguishes “a transaction 

⚠ WARNING: No prompts logged.
» ]8;id=39404;https://deepeval.com/docs/evaluation-prompts\Log prompts]8;;\ to evaluate and optimize your prompt templates and models.

================================================================================

Test run saved at /content/legai_eval_fixed50_prompt_v2/test_run_20260928_205613.json

✓ Evaluation completed 🎉! (time taken: 771.66s | token cost: 0.7386937500000001 USD)
» Test Results (50 total tests):
   » Pass Rate: 14.0% | Passed: 7 | Failed: 43

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

# 16. SHOW SAVED RESULT FILES

In [ ]:
print("\nRETRIEVAL EVALUATION FILES:")
!find /content/legai_eval_retrieval_v2 -maxdepth 1 -type f -print

print("\nANSWER EVALUATION FILES:")
!find /content/legai_eval_answers_v2 -maxdepth 1 -type f -print